In [2]:
all_dataset_names = [
    "AbdomenUS",
    "Acdc",
    "BAGLS",
    "Bbbc010",
    "BkaiIgh",
    "BriFiSeg",
    "Btcv",
    "Busi",
    "CellNuclei",
    "Chaos",
    "ChaseDB1",
    "Chuac",
    "Covid19Radio",
    "CovidQUEx",
    "CystoFluid",
    "Dca1",
    "Deepbacs",
    "Drive",
    "DynamicNuclear",
    "EMSegmentation",
    "FHPsAOP",
    "Idrib",
    "Isic2016",
    "Isic2018",
    "Kvasir",
    "M2caiSeg",
    "MmWhsMr",
    "Monusac",
    "MosMedPlus",
    "Nuclei",
    "Nuset",
    "Pandental",
    "PolypGen",
    "Promise12",
    "RoboTool",
    "TnbcNuclei",
    "UltrasoundNerve",
    "USforKidney",
    "UwSkinCancer",
    "Wbc",
    "Yeaz",
]

def find_latest_runs_per_dataset_filtered_by_conditions(experiments_csv_path, conditions):
    import pandas as pd

    all_experiments = pd.read_csv(experiments_csv_path)

    matching_experiments = all_experiments
    for column_name, column_value in conditions.items():
        matching_experiments = matching_experiments[matching_experiments[column_name] == column_value]

    condition_description = ", ".join(f"{column_name} == {column_value}" for column_name, column_value in conditions.items())

    print(f"Conditions: {condition_description}")
    print(f"Total matching entries in csv: {len(matching_experiments)}")

    if matching_experiments.empty:
        return matching_experiments

    matching_experiments = matching_experiments.sort_values("date")
    latest_run_per_dataset = matching_experiments.groupby("dataset").tail(1)
    latest_run_per_dataset = latest_run_per_dataset.sort_values("dataset")

    # display(latest_run_per_dataset)

    return latest_run_per_dataset


def check_missing_datasets_for_experiment(filtered_experiments, all_dataset_names):
    datasets_present = set(filtered_experiments["dataset"].unique())
    datasets_missing = sorted(set(all_dataset_names) - datasets_present)

    print(f"Total datasets: {len(all_dataset_names)}")
    print(f"Datasets present: {len(datasets_present)}")
    print(f"Datasets missing: {len(datasets_missing)}")
    if datasets_missing:
        print(f"Missing datasets: {datasets_missing}")

    return datasets_missing

In [ ]:
latest_runs = find_latest_runs_per_dataset_filtered_by_conditions(
    "experiments/experiments_large.csv",
    {
        # "hypothesis": "Depth-wise separable layers (18-36-72) + attention gate does not lose much in Dice.",
        # "hypothesis": "Depth-wise separable layers (16-32-64) + attention gate does not lose much in Dice.",
        "hypothesis": "Depth-wise separable layers (12-24-48) + triple convolution per block + attention gate does not lose much in Dice.",
        # "hypothesis": "Depth-wise separable layers (16-32-64) with three convolutions per block does not lose much in Dice.",
        # "hypothesis": "Depth-wise separable layers (24-48-96) does not lose much in Dice.",
        # "architecture_use_residual_connections": True,
        # "date": "2026-07-19",
    }
)

check_missing_datasets_for_experiment(latest_runs, all_dataset_names)

Conditions: hypothesis == Depth-wise separable layers (16-32-64) + triple convolution per block + attention gate does not lose much in Dice.
Total matching entries in csv: 40
Total datasets: 41
Datasets present: 40
Datasets missing: 1
Missing datasets: ['BAGLS']


['BAGLS']

In [4]:
# latest_runs = find_latest_runs_per_dataset_filtered_by_conditions(
#     "experiments/experiments_large.csv",
#     {
#         "date": "2026-07-19",
#         "architecture_use_residual_connections": False,
#         "hypothesis": "Depth-wise separable layers (16-32-64) with three convolutions per block does not lose much in Dice.",
#     }
# )

# # Extract run_ids from the DataFrame
# if isinstance(latest_runs, pd.DataFrame):
#     latest_runs = latest_runs['run_id'].tolist()

# print(f"Matched run_ids: {latest_runs}")
# print(f"Total matched: {len(latest_runs)}\n")

# paths = ["experiments/experiments_large.csv", "experiments/experiments.csv"]

# for path in paths:
#     table = pd.read_csv(path, engine="python", on_bad_lines="warn")
#     table.to_csv(path, index=False)

# new_hypothesis = "Depth-wise separable layers (16-32-64) + triple convolution per block does not lose much in Dice."
# new_notes = "Depth-wise separable layers (16-32-64) + triple convolution"

# for path in paths:
#     table = pd.read_csv(path)
#     rows_to_update = table["run_id"].isin(latest_runs)
#     table.loc[rows_to_update, "hypothesis"] = new_hypothesis
#     table.loc[rows_to_update, "notes"] = new_notes
#     table.to_csv(path, index=False)
#     print(f"{path}: updated {int(rows_to_update.sum())} rows")

In [5]:
# import pandas as pd

# hypothesis_to_delete = "Depth-wise separable layers (18-36-72) + attention gate does not lose much in Dice."

# paths = ["experiments/experiments_large.csv", "experiments/experiments.csv"]

# for path in paths:
#     table = pd.read_csv(path)
#     rows_to_keep = table["hypothesis"] != hypothesis_to_delete
#     table = table[rows_to_keep]
#     table.to_csv(path, index=False)
#     deleted_count = (~rows_to_keep).sum()
#     print(f"{path}: deleted {int(deleted_count)} rows")

In [6]:
import pandas as pd
from pathlib import Path


EXPERIMENTS_LARGE_CSV_PATH = Path("experiments/experiments_large.csv")


def compare_last_two_runs_per_dataset_for_hypothesis(hypothesis_text):
    all_experiments = pd.read_csv(EXPERIMENTS_LARGE_CSV_PATH)

    matching_experiments = all_experiments[
        all_experiments["dataset"].isin(all_dataset_names) &
        (all_experiments["hypothesis"].str.strip() == hypothesis_text)
    ].copy()

    if matching_experiments.empty:
        print(f"No experiments found for hypothesis: {hypothesis_text!r}")
        return

    matching_experiments = matching_experiments.sort_values("run_id")
    last_two_runs_per_dataset = matching_experiments.groupby("dataset").tail(2)

    print(f"Hypothesis: {hypothesis_text!r}\n")
    print(f"{'Dataset':<25} {'Run ID':<25} {'Mean Val Dice':<16} {'Std Val Dice'}")
    print("-" * 85)

    for dataset_name in all_dataset_names:
        dataset_runs = last_two_runs_per_dataset[
            last_two_runs_per_dataset["dataset"] == dataset_name
        ].sort_values("run_id")

        if dataset_runs.empty:
            print(f"{dataset_name:<25} {'(no runs found)'}")
            print()
            continue

        for _, run_row in dataset_runs.iterrows():
            print(
                f"{run_row['dataset']:<25} {run_row['run_id']:<25} "
                f"{run_row['mean_val_dice']:<16.4f} {run_row['std_val_dice']:.4f}"
            )
        if len(dataset_runs) == 2:
            dice_change = dataset_runs.iloc[1]["mean_val_dice"] - dataset_runs.iloc[0]["mean_val_dice"]
            print(f"{'':25} {'Δ dice':<25} {dice_change:+.4f}")
        print()


compare_last_two_runs_per_dataset_for_hypothesis("No hypothesis -  data augmentation + instance norm.")

Hypothesis: 'No hypothesis -  data augmentation + instance norm.'

Dataset                   Run ID                    Mean Val Dice    Std Val Dice
-------------------------------------------------------------------------------------
AbdomenUS                 2026_07_13_01_47_49_1740137 0.5653           0.0606
AbdomenUS                 2026_07_13_21_08_43_1742118 0.5656           0.1020
                          Δ dice                    +0.0003

Acdc                      2026_07_13_01_47_40_1740133 0.8567           0.0073
Acdc                      2026_07_13_21_08_14_1742114 0.8577           0.0180
                          Δ dice                    +0.0010

BAGLS                     (no runs found)

Bbbc010                   2026_07_13_01_47_37_1740138 0.8219           0.0276
Bbbc010                   2026_07_13_21_08_43_1742119 0.7895           0.0356
                          Δ dice                    -0.0324

BkaiIgh                   2026_07_13_01_47_39_1740139 0.7295           